# 🌌 NeoWatch — Phase 1: Data Collection & NASA API Ingestion

**Objective**: Ingest Near-Earth Object (NEO) astronomical data via the NASA NeoWs REST API, handle rate limiting, parse nested JSON structures, flatten the data, and export to `data/raw_asteroid_data.csv` (**Checkpoint 1**).

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to sys.path
project_root = Path(os.path.abspath('')).parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
from src.config import NASA_API_KEY, RAW_DATA_PATH
from src.api_client import NASAClient

print(f"NASA API Key Configured: {'Yes (' + NASA_API_KEY[:6] + '...)' if NASA_API_KEY else 'No'}")

## 1. Test NASA NeoWs API with a 7-Day Chunk

In [ ]:
client = NASAClient()

# Sample 7-day query
sample_json = client.fetch_feed_chunk(start_date="2023-01-01", end_date="2023-01-07")
print(f"Element count returned: {sample_json.get('element_count')}")

# Parse and flatten
sample_records = client.parse_feed_json(sample_json)
sample_df = pd.DataFrame(sample_records)
sample_df.head(3)

## 2. Ingest Multi-Month / Multi-Year Dataset

In [ ]:
# Fetch 1-year historical data
start_date = "2023-01-01"
end_date = "2024-01-01"

df_raw = client.fetch_date_range(start_date, end_date, delay_between_calls=0.25)
print(f"Dataset shape: {df_raw.shape}")
df_raw.head()

## 3. Preliminary Inspection & Checkpoint 1 Validation

In [ ]:
# Inspect column data types and missing values
print(df_raw.info())

# Class distribution
target_counts = df_raw['is_potentially_hazardous_asteroid'].value_counts()
print("\n--- Target Distribution ---")
print(target_counts)
print(f"Hazardous Ratio: {df_raw['is_potentially_hazardous_asteroid'].mean()*100:.2f}%")

# Save to CSV
client.save_to_csv(df_raw, RAW_DATA_PATH)
print(f"\n🎯 Checkpoint 1 Complete: Saved to {RAW_DATA_PATH}")